# 🏡 San Antonio Automated Apartment & Townhome Hunter (v3 - Complete Standalone Edition)
This complete notebook executes the entire end-to-end multi-criteria pipeline in place:
1. **Walmart-Excluded Grocery Proximity KD-Tree:** Skips Walmart and targets H-E-B, Trader Joe's, Sprouts, Whole Foods, and Central Market.
2. **Edwards Aquifer Water Softener / RO Detector:** Detects water softeners, softener loops, and reverse-osmosis filtration.
3. **Hard Flooring & Cat-Friendly Scorer:** Rewards 100% luxury vinyl plank / hardwood / tile and checks cat policies.
4. **Regex Junk Fee Parser:** Extracts itemized monthly corporate fees and utility inclusions.
5. **Spatial Noise KD-Trees:** Calculates distance matrices against OpenStreetMap railways and San Antonio International Airport (SAT) approach vectors.

In [ ]:
# @title Step 0: Install Required Packages
!pip install -q homeharvest osmnx geopandas folium shapely scipy requests google-genai pandas pyyaml

In [ ]:
# @title Step 1: Configuration & Setup
import os
import re
import math
import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import folium
from shapely.geometry import Point
from scipy.spatial import cKDTree
from homeharvest import scrape_property

CONFIG = {
    'workplace': {'name': 'Southwest Research Institute (SwRI)', 'coords': (29.4485, -98.6105)},
    'costco_locations': [
        {'name': 'Costco NW (5611 UTSA Blvd)', 'coords': (29.5762, -98.6001)},
        {'name': 'Costco West (191 W Loop 1604 S)', 'coords': (29.4316, -98.7115)},
        {'name': 'Costco North Central (1201 N Loop 1604 E)', 'coords': (29.6095, -98.4716)},
        {'name': 'Costco NE Selma (15330 IH-35 N)', 'coords': (29.5794, -98.3188)}
    ],
    'filters': {'location': 'San Antonio, TX', 'min_rent': 1200, 'max_rent': 2200, 'min_beds': 2, 'min_baths': 1.5, 'past_days': 30},
    'cutoffs': {'require_in_unit_laundry_and_dishwasher': True, 'strict_appliance_text_match': False, 'max_swri_commute_mins': 28.0, 'max_grocery_dist_m': 2400.0, 'max_park_trail_dist_m': 1600.0},
    'weights': {
        'commute_score': 0.18,
        'grocery_score': 0.12,
        'park_trail_score': 0.12,
        'pollution_highway_score': 0.08,
        'modern_interior_score': 0.12,
        'garage_bike_score': 0.08,
        'costco_score': 0.08,
        'water_softener_score': 0.06,
        'pet_flooring_score': 0.06,
        'effective_rent_score': 0.05,
        'noise_buffer_score': 0.05
    }
}
print('Configuration loaded!')

In [ ]:
# @title Step 2: Build Spatial KD-Trees (Groceries, Parks, Highways, Railways)
print('Fetching OpenStreetMap geospatial layers...')
loc = CONFIG['filters']['location']

def get_osm_tree(place, tags):
    gdf = ox.features_from_place(place, tags=tags).to_crs(epsg=32614)
    xy = np.array([(geom.centroid.x, geom.centroid.y) for geom in gdf.geometry])
    names = gdf['name'].fillna('Unnamed').tolist() if 'name' in gdf.columns else ['Unnamed']*len(xy)
    return cKDTree(xy), names

TREES = {
    'grocery': get_osm_tree(loc, {'shop': ['supermarket', 'grocery', 'greengrocer']}),
    'park': get_osm_tree(loc, {'leisure': ['park', 'nature_reserve'], 'highway': ['cycleway', 'path', 'footway']}),
    'highway': get_osm_tree(loc, {'highway': ['motorway', 'trunk']}),
    'railway': get_osm_tree(loc, {'railway': 'rail'})
}
print('All spatial KD-Trees successfully built!')

In [ ]:
# @title Step 3: Define All v3 Criteria Scoring Functions
CIRCUITY_FACTOR = 1.32
WALK_SPEED_M_MIN = 80.0
BIKE_SPEED_M_MIN = 250.0

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1))*math.cos(math.radians(lat2))*math.sin(dlon/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def criterion_required_appliances_gatekeeper(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    year_built = row.get('year_built', np.nan)
    shared_only_flags = ['shared laundry', 'laundromat on site', 'no washer', 'no laundry in unit', 'coin-operated laundry']
    if any(flag in desc for flag in shared_only_flags):
        return {'appliances_status': 'REJECT: Shared/No In-Unit Laundry', 'passes_appliance_gate': False}
    in_unit_terms = ['in-unit', 'in unit', 'washer/dryer', 'washer and dryer', 'w/d included', 'w/d connections', 'laundry room', 'utility room', 'full-size washer']
    has_in_unit = any(t in desc for t in in_unit_terms)
    has_dishwasher = 'dishwasher' in desc or ('stainless' in desc and 'appliances' in desc) or (pd.notna(year_built) and year_built >= 2005)
    return {'appliances_status': 'Verified (In-Unit W/D + Dishwasher)' if (has_in_unit and has_dishwasher) else 'Likely (Verify W/D in Photos)', 'passes_appliance_gate': True}

def criterion_garage_and_bike_storage(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    prop_type = str(row.get('style', '') or row.get('type', '') or '').lower()
    parking_spaces = row.get('parking_garage', np.nan)
    score, features = 30.0, []
    if pd.notna(parking_spaces) and float(parking_spaces) >= 1:
        score += 45.0
        features.append(f'{int(float(parking_spaces))}-Car Garage')
    elif any(k in desc for k in ['attached garage', 'private garage', '1-car garage', '2-car garage', 'two-car garage', 'single car garage']):
        score += 45.0
        features.append('Private/Attached Garage')
    elif 'carport' in desc or 'covered parking' in desc:
        score += 20.0
        features.append('Covered/Carport')
    if any(k in desc for k in ['bike storage', 'bicycle storage', 'bike rack', 'storage unit', 'storage closet']):
        score += 25.0
        features.append('Dedicated Bike/Gear Storage')
    return {'garage_bike_features': ', '.join(features) if features else 'Standard Surface Parking', 'garage_bike_score': round(float(np.clip(score, 0.0, 100.0)), 1)}

def criterion_costco_proximity(row, ctx):
    lat, lon = row['latitude'], row['longitude']
    best_name, min_road_km = '', 999.0
    for c in ctx['config']['costco_locations']:
        clat, clon = c['coords']
        road_km = haversine_km(lat, lon, clat, clon) * 1.28
        if road_km < min_road_km:
            min_road_km, best_name = road_km, c['name']
    est_drive_mins = (min_road_km / 48.0) * 60.0 + 2.0
    score = np.clip(100 - ((est_drive_mins - 7.0) / (25.0 - 7.0)) * 100, 0, 100)
    return {'nearest_costco': best_name, 'costco_drive_mins': round(est_drive_mins, 1), 'costco_score': round(float(score), 1)}

def criterion_grocery_proximity(row, ctx):
    tree, names = ctx['trees']['grocery']
    distances, indices = tree.query([row['utm_x'], row['utm_y']], k=min(5, len(names)))
    if not isinstance(indices, np.ndarray):
        indices, distances = [indices], [distances]
    best_idx, best_dist, best_name = indices[0], distances[0], names[indices[0]]
    for idx, dist in zip(indices, distances):
        if 'walmart' not in names[idx].lower():
            best_idx, best_dist, best_name = idx, dist, names[idx]
            break
    net_m = best_dist * CIRCUITY_FACTOR
    score = np.clip(100 - ((net_m - 400) / (2400 - 400)) * 100, 0, 100)
    return {'nearest_grocery': best_name, 'grocery_dist_m': round(net_m), 'grocery_walk_mins': round(net_m / WALK_SPEED_M_MIN, 1), 'grocery_bike_mins': round(net_m / BIKE_SPEED_M_MIN, 1), 'grocery_score': round(float(score), 1)}

def criterion_park_trail_proximity(row, ctx):
    tree, names = ctx['trees']['park']
    dist_euc, idx = tree.query([row['utm_x'], row['utm_y']])
    net_m = dist_euc * 1.25
    score = np.clip(100 - ((net_m - 250) / (1600 - 250)) * 100, 0, 100)
    return {'nearest_park_trail': names[idx], 'park_dist_m': round(net_m), 'park_walk_mins': round(net_m / WALK_SPEED_M_MIN, 1), 'park_trail_score': round(float(score), 1)}

def criterion_swri_rush_hour_commute(row, ctx):
    lat, lon = row['latitude'], row['longitude']
    swri_lat, swri_lon = ctx['config']['workplace']['coords']
    road_km = haversine_km(lat, lon, swri_lat, swri_lon) * 1.30
    avg_speed = 28.0 if lon < -98.68 else (50.0 if lon > -98.58 else 40.0)
    est_mins = (road_km / avg_speed) * 60.0 + 3.0
    score = np.clip(100 - ((est_mins - 8.0) / (30.0 - 8.0)) * 100, 0, 100)
    return {'swri_rush_hour_mins': round(est_mins, 1), 'commute_score': round(float(score), 1)}

def criterion_pollution_and_highway_buffer(row, ctx):
    tree, _ = ctx['trees']['highway']
    hwy_dist_m, _ = tree.query([row['utm_x'], row['utm_y']])
    score = 15.0 if hwy_dist_m < 100 else (50.0 + ((hwy_dist_m - 100) / 150.0) * 35.0 if hwy_dist_m < 250 else np.clip(85.0 + ((hwy_dist_m - 250) / 500.0) * 15.0, 85.0, 100.0))
    return {'highway_buffer_m': round(hwy_dist_m), 'pollution_highway_score': round(float(score), 1)}

def criterion_modern_appliances_and_open_floorplan(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    score = 50.0 + sum([18 for k in ['open floor', 'open-concept', 'open concept'] if k in desc]) + sum([15 for k in ['stainless steel', 'stainless appliances'] if k in desc])
    return {'modern_interior_score': round(float(np.clip(score, 10.0, 100.0)), 1)}

def criterion_water_softener_and_filtration(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    score = 40.0 + sum([40 for k in ['water softener', 'softener loop'] if k in desc]) + sum([25 for k in ['reverse osmosis', 'ro system'] if k in desc])
    return {'water_softener_score': round(float(np.clip(score, 10.0, 100.0)), 1)}

def criterion_pet_friendly_and_hard_flooring(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    score = 60.0 + (25.0 if any(t in desc for t in ['vinyl plank', 'lvp', 'hardwood', 'no carpet']) else 0.0) + (15.0 if any(t in desc for t in ['cat friendly', 'pets allowed']) else 0.0)
    return {'pet_flooring_score': round(float(np.clip(score, 10.0, 100.0)), 1)}

def criterion_advanced_junk_fee_parser(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    fees = sum([int(m) for m in re.findall(r'\$(\d+)\s*(?:/mo)?\s*(?:valet|trash|waste|package|tech|fee)', desc)])
    score = np.clip(100.0 - min(fees, 50.0) + (10.0 if 'water included' in desc else 0.0), 30.0, 100.0)
    return {'extracted_monthly_fees': fees, 'effective_rent_score': round(float(score), 1)}

def criterion_spatial_noise_buffer(row, ctx):
    rail_dist = ctx['trees']['railway'][0].query([row['utm_x'], row['utm_y']])[0] if 'railway' in ctx['trees'] else 9999
    sat_dist = haversine_km(row['latitude'], row['longitude'], 29.5338, -98.4698)
    score = 90.0 - (40.0 if rail_dist < 200 else (15.0 if rail_dist < 500 else 0.0)) - (25.0 if sat_dist < 6.0 else 0.0)
    return {'noise_buffer_score': round(float(np.clip(score, 20.0, 100.0)), 1)}

ACTIVE_CRITERIA = [
    criterion_required_appliances_gatekeeper, criterion_garage_and_bike_storage,
    criterion_costco_proximity, criterion_grocery_proximity, criterion_park_trail_proximity,
    criterion_swri_rush_hour_commute, criterion_pollution_and_highway_buffer,
    criterion_modern_appliances_and_open_floorplan, criterion_water_softener_and_filtration,
    criterion_pet_friendly_and_hard_flooring, criterion_advanced_junk_fee_parser, criterion_spatial_noise_buffer
]
print('All criteria functions successfully registered!')

In [ ]:
# @title Step 4: Execute Scraper, Score Pipeline & Export CSV + Map
print('Scraping listings in San Antonio...')
raw = scrape_property(location=CONFIG['filters']['location'], listing_type='for_rent', past_days=CONFIG['filters']['past_days'])
df = raw.dropna(subset=['latitude', 'longitude', 'list_price']).copy()
f = CONFIG['filters']
df = df[(df['list_price'] >= f['min_rent']) & (df['list_price'] <= f['max_rent']) & (df['beds'].fillna(0) >= f['min_beds']) & (df['full_baths'].fillna(0) >= 1)].copy()

gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs='EPSG:4326').to_crs(epsg=32614)
df['utm_x'], df['utm_y'] = gdf.geometry.x, gdf.geometry.y

ctx = {'config': CONFIG, 'trees': TREES}
results = []
weights = CONFIG['weights']

for _, row in df.iterrows():
    m = {}
    for fn in ACTIVE_CRITERIA:
        m.update(fn(row, ctx))
    if m.get('passes_appliance_gate', True):
        m['composite_score'] = round(sum(m.get(k, 50.0) * w for k, w in weights.items()), 1)
        results.append(m)
    else:
        results.append(None)

scored_df = pd.concat([df.reset_index(drop=True), pd.DataFrame(results)], axis=1)
scored_df = scored_df.dropna(subset=['composite_score']).sort_values(by='composite_score', ascending=False).reset_index(drop=True)

os.makedirs('outputs', exist_ok=True)
scored_df.to_csv('outputs/san_antonio_ranked_rentals_v3.csv', index=False)
print(f'Pipeline finished! Exported {len(scored_df)} top-ranked properties to outputs/san_antonio_ranked_rentals_v3.csv')
scored_df[['composite_score', 'formatted_address', 'list_price', 'beds', 'full_baths', 'nearest_grocery', 'nearest_costco']].head(15)